# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician participating in a highly rigorous math competition. "
    "Solve the problem step-by-step. You must first write out a detailed reasoning trace. "
    "After completing your reasoning, place your final answer inside \\boxed{}. "
    "CRITICAL: If the problem asks for multiple answers (indicated by multiple [ANS] placeholders), "
    "you MUST calculate all of them and separate them by commas inside a SINGLE \\boxed{}, "
    "in the exact order they were requested. For example: \\boxed{3, 7, 42}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and the provided answer choices carefully. "
    "Think step-by-step to derive the correct solution, then compare your solution to the options. "
    "If your calculated answer does not perfectly match any option, you must choose the option that "
    "is mathematically closest or most logically intended. "
    "CRITICAL: You must conclude your response by outputting ONLY the single letter of your chosen option "
    "inside \\boxed{}, for example: \\boxed{C}. Never leave a response without a boxed letter, regardless of your confidence."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    trust_remote_code=True,
)

# 1. Update Sampling Params to generate multiple outputs per prompt
SC_SAMPLES = 7 # Increased from 5 for stronger majority voting

sampling_params = SamplingParams(
    n=SC_SAMPLES,              
    max_tokens=4096,
    temperature=0.5,           # Lowered from 0.7 to reduce logical hallucinations
    top_p=0.95,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

(EngineCore pid=429) 

INFO 05-07 22:28:30 [core.py:299] init engine (profile, create kv cache, warmup model) took 103.69 s (compilation: 24.13 s)


(EngineCore pid=429) 

INFO 05-07 22:28:30 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [6]:
import csv

# 1. Initialize the CSV with headers (Write mode)
OUTPUT_CSV_PATH = "results/submission.csv"
out_path = Path(OUTPUT_CSV_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

# with open(out_path, "w", newline="", encoding="utf-8") as f:
#     writer = csv.writer(f, quoting=csv.QUOTE_ALL)
#     writer.writerow(["id", "response"])

In [7]:
import re
from collections import Counter
from pathlib import Path

# Build prompts for the dataset
prompts = []
for item in data[:100]: # Run on full data
    system, user = build_prompt(item["question"], item.get("options"))
    
    # --- Free-Form Few Shot ---
    few_shot_free_user = "If $f(x)=4x^2+x+2$, find the following:\n(a) $f(3)=$ [ANS]\n(b) $f(-3)=$ [ANS]"
    few_shot_free_assistant = (
        "Let's break this down step-by-step.\n"
        "First, for (a):\n$f(3) = 4(3)^2 + 3 + 2 = 36 + 5 = 41$\n\n"
        "Second, for (b):\n$f(-3) = 4(-3)^2 + (-3) + 2 = 36 - 1 = 35$\n\n"
        "The problem requests multiple answers. I will format them in a single boxed array.\n"
        "The final answer is \\boxed{41, 35}."
    )

    # --- MCQ Few Shot ---
    few_shot_mcq_user = (
        "Evaluate the integral $\\int 2x \\, dx$.\n\n"
        "Options:\n"
        "A. $x^2 + C$\n"
        "B. $2x^2 + C$\n"
        "C. $\\frac{1}{2}x^2 + C$\n"
        "D. $2 + C$"
    )
    few_shot_mcq_assistant = (
        "Let's think step-by-step.\n"
        "We need to find the indefinite integral of the function $f(x) = 2x$.\n"
        "Using the power rule for integration, $\\int x^n \\, dx = \\frac{x^{n+1}}{n+1} + C$.\n"
        "Therefore, $\\int 2x \\, dx = 2 \\left( \\frac{x^2}{2} \\right) + C = x^2 + C$.\n\n"
        "Now, I will match this result with the given options.\n"
        "Option A is $x^2 + C$. This matches our calculated result exactly.\n"
        "Option B, C, and D are incorrect.\n"
        "Therefore, the correct option is A.\n"
        "The final answer is \\boxed{A}."
    )

    # Build the message history dynamically based on question type
    messages = [{"role": "system", "content": system}]
    
    if not item.get("options"):
        # Inject Free-Form Example
        messages.append({"role": "user", "content": few_shot_free_user})
        messages.append({"role": "assistant", "content": few_shot_free_assistant})
    else:
        # Inject MCQ Example
        messages.append({"role": "user", "content": few_shot_mcq_user})
        messages.append({"role": "assistant", "content": few_shot_mcq_assistant})
        
    messages.append({"role": "user", "content": user})

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# 2. Generate
print(f"Generating {SC_SAMPLES} responses per question for {len(prompts)} questions...")

# 2. Process in chunks
CHUNK_SIZE = 50  # Process 50 questions at a time

print(f"Starting generation for {len(prompts)} prompts in chunks of {CHUNK_SIZE}...")

responses = []
# Iterate through the prompts in steps of CHUNK_SIZE
for i in range(0, len(prompts), CHUNK_SIZE):
    chunk_prompts = prompts[i : i + CHUNK_SIZE]
    chunk_data = data[i : i + CHUNK_SIZE] # Keep the IDs aligned!
    
    # Generate for this specific chunk
    outputs = llm.generate(chunk_prompts, sampling_params=sampling_params)
    
    # # PRIVATE DATASET: Open CSV in APPEND mode ("a") to add the new rows without overwriting
    # with open(out_path, "a", newline="", encoding="utf-8") as f:
    #     writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        
    #     # Process Majority Voting for the chunk
    #     for item, out in zip(chunk_data, outputs):
    #         candidate_texts = [o.text.strip() for o in out.outputs]
            
    #         candidate_answers = []
    #         for text in candidate_texts:
    #             match = re.search(r"\\boxed\{(.*?)\}", text, flags=re.DOTALL)
    #             if match:
    #                 candidate_answers.append(match.group(1).strip())
            
    #         if candidate_answers:
    #             majority_answer = Counter(candidate_answers).most_common(1)[0][0]
    #             final_full_text = next((t for t in candidate_texts if f"\\boxed{{{majority_answer}}}" in t), candidate_texts[0])
    #         else:
    #             final_full_text = candidate_texts[0]
            
    #         # Write the row immediately to the disk
    #         writer.writerow([item["id"], final_full_text])

    # PUBLIC DATASET: Append to responses instead of csv        
    # Process Majority Voting for the chunk
    for item, out in zip(chunk_data, outputs):
        candidate_texts = [o.text.strip() for o in out.outputs]
        
        candidate_answers = []
        for text in candidate_texts:
            match = re.search(r"\\boxed\{(.*?)\}", text, flags=re.DOTALL)
            if match:
                candidate_answers.append(match.group(1).strip())
        
        if candidate_answers:
            majority_answer = Counter(candidate_answers).most_common(1)[0][0]
            responses.append(next((t for t in candidate_texts if f"\\boxed{{{majority_answer}}}" in t), candidate_texts[0]))
        else:
            responses.append(candidate_texts[0])
            
    print(f"Saved chunk. Processed {min(i + CHUNK_SIZE, len(prompts))}/{len(prompts)} questions.")

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating 7 responses per question for 943 questions...
Starting generation for 943 prompts in chunks of 50...


Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/350 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved chunk. Processed 800/943 questions.


Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/350 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved chunk. Processed 900/943 questions.


Rendering prompts:   0%|          | 0/43 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/301 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved chunk. Processed 943/943 questions.


### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [15]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   16 /   38  (42.11%)
  Free-form  :   32 /   62  (51.61%)
  Overall    :   48 /  100  (48.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [16]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [17]:
# Extract failed MCQs
failed_mcqs = [r for r in results if r["is_mcq"] and not r["correct"]]

print(f"Total failed MCQs: {len(failed_mcqs)}\n")
for f in failed_mcqs[:5]: # Let's look at the first 5
    print("="*40)
    print(f"Question ID: {f['id']}")
    print(f"Gold Answer: {f['gold']}")
    
    # Extract just the boxed part from the model's response to save screen space
    import re
    match = re.search(r"\\boxed\{(.*?)\}", f["response"], flags=re.DOTALL)
    pred_box = match.group(1) if match else "No box found"
    
    print(f"Model Boxed: {pred_box}")

Total failed MCQs: 22

Question ID: 1
Gold Answer: F
Model Boxed: No box found
Question ID: 10
Gold Answer: E
Model Boxed: No box found
Question ID: 11
Gold Answer: G
Model Boxed: No box found
Question ID: 13
Gold Answer: J
Model Boxed: No box found
Question ID: 14
Gold Answer: F
Model Boxed: No box found
